In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

In [ ]:
LOCAL_DIR = "/content/netguard_data"
OUT_DIR = os.path.join(LOCAL_DIR, "processed")

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Aegis-CyberSec-Guard"
DRIVE_OUT = os.path.join(DRIVE_ROOT, "netguard", "dataV1")

In [ ]:
KAGGLE_DATASET = "chethuhn/network-intrusion-dataset"

In [ ]:
# Columns that identify a flow but must NOT be used as model features (leakage / not
# generalizable — IPs and timestamps memorize the capture, not attack behavior).
NON_FEATURE_COLUMNS = ["Flow ID", "Source IP", "Source Port", "Destination IP", "Timestamp"]

In [ ]:
# Destination Port IS kept as a feature — attacks often target specific ports (21, 22,80),
# which is a legitimate signal, not leakage the way Source IP would be.

LABEL_COLUMN_CANDIDATES = ["Label", "label", " Label"]  # whitespace issue handled by strip

In [ ]:
# Classes below this count get merged into "Other" so stratified splitting is possible.
MIN_CLASS_COUNT = 200
TEST_SIZE = 0.10
VAL_SIZE = 0.15   # of the remaining after test split
SEED = 3407


In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Local work dir: {LOCAL_DIR}")
print(f"Final tensors will be copied to: {DRIVE_OUT}")


Local work dir: /content/netguard_data
Final tensors will be copied to: /content/drive/MyDrive/Aegis-CyberSec-Guard/netguard/dataV1


## Import ML Libraries

In [ ]:
import re
import json
import glob

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder

In [ ]:
np.random.seed(SEED)

## Download Dataset by kagglehub

In [ ]:
# kagglehub caches to local disk (not Drive) by default.
import kagglehub

In [ ]:
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f"Downloaded to: {dataset_path}")

Using Colab cache for faster access to the 'network-intrusion-dataset' dataset.
Downloaded to: /kaggle/input/network-intrusion-dataset


In [ ]:
csv_files = sorted(glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True))

In [ ]:
print(f"\nFound {len(csv_files)} CSV files:")
for f in csv_files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f):55s} {size_mb:8.1f} MB")



Found 8 CSV files:
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv            77.1 MB
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv        76.9 MB
  Friday-WorkingHours-Morning.pcap_ISCX.csv                   58.3 MB
  Monday-WorkingHours.pcap_ISCX.csv                          176.9 MB
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv     83.1 MB
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv      52.0 MB
  Tuesday-WorkingHours.pcap_ISCX.csv                         135.1 MB
  Wednesday-workingHours.pcap_ISCX.csv                       225.2 MB


In [ ]:
# Check if Dataset Import wroks
assert len(csv_files) >= 7, (
    f"Expected ~8 daily CSV files (one per capture day), found {len(csv_files)}. "
    "Loading only a subset would mean missing most attack types — Monday alone is "
    "100% benign traffic. Inspect dataset_path before continuing."
)

##  Inspect ONE raw file before touching all of them

In [ ]:
# This is where the whitespace-in-column-names issue becomes visible directly, instead
# of being discovered later as a silent KeyError.
sample_df = pd.read_csv(csv_files[0], nrows=5)

In [ ]:
# Print  Sample df file
print(f"File: {os.path.basename(csv_files[0])}")
print(f"Shape (sample): {sample_df.shape}")
print("\nColumn names as they REALLY are (repr shows hidden whitespace):")
for c in sample_df.columns[:10]:
    print(f"  {c!r}")
print("  ...")

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Shape (sample): (5, 79)

Column names as they REALLY are (repr shows hidden whitespace):
  ' Destination Port'
  ' Flow Duration'
  ' Total Fwd Packets'
  ' Total Backward Packets'
  'Total Length of Fwd Packets'
  ' Total Length of Bwd Packets'
  ' Fwd Packet Length Max'
  ' Fwd Packet Length Min'
  ' Fwd Packet Length Mean'
  ' Fwd Packet Length Std'
  ...


In [ ]:
label_col_raw = [c for c in sample_df.columns if c.strip().lower() == "label"]
print(f"\nRaw label column name: {label_col_raw!r}")
print(f"Sample label values: {sample_df[label_col_raw[0]].unique() if label_col_raw else 'NOT FOUND'}")


Raw label column name: [' Label']
Sample label values: ['BENIGN']


##  Load and Concatenate all 8 files, standardizing columns per file
#

Column names are stripped BEFORE concatenation. If different files had inconsistent spacing (a real risk with this dataset across its multiple source files), concatenating first and stripping after could silently create duplicate columns with different names for the same field. Strip first, then concat, so there is only ever one "Label" column.

In [ ]:
dfs = []
for f in csv_files:
    df = pd.read_csv(f, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    dfs.append(df)
    print(f"  {os.path.basename(f):55s} {len(df):>10,} rows, {len(df.columns)} columns")

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv           225,745 rows, 79 columns
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv       286,467 rows, 79 columns
  Friday-WorkingHours-Morning.pcap_ISCX.csv                  191,033 rows, 79 columns
  Monday-WorkingHours.pcap_ISCX.csv                          529,918 rows, 79 columns
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv    288,602 rows, 79 columns
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv     170,366 rows, 79 columns
  Tuesday-WorkingHours.pcap_ISCX.csv                         445,909 rows, 79 columns
  Wednesday-workingHours.pcap_ISCX.csv                       692,703 rows, 79 columns


In [ ]:
full_df = pd.concat(dfs, axis=0, ignore_index=True)
del dfs  # free memory before the heavier cleaning steps

In [ ]:
print(f"\nConcatenated: {len(full_df):,} rows, {len(full_df.columns)} columns")
assert "Label" in full_df.columns, "Label column missing after stripping whitespace"


Concatenated: 2,830,743 rows, 79 columns


## Clean: Infinity, NaN, duplicates

Known issue: Flow Bytes/s and Flow Packets/s divide by flow duration, which can be zero for very short flows, producing +/-Infinity. These are not rare edge cases to special-case individually — replace Infinity with NaN globally, then drop.

In [ ]:
# Known CICIDS2017 defect: "Fwd Header Length" appears twice in the raw CSVs.
# Pandas auto-renames the second occurrence to "Fwd Header Length.
print(f"Columns before checking for drop: {len(full_df.columns)}")
print("full_df.columns (repr):")
for col in full_df.columns:
    print(f"  {col!r}")
#  Drop the duplicate before it reaches the model.
if "Fwd Header Length.1" in full_df.columns:
    dup_check = (full_df["Fwd Header Length"] == full_df["Fwd Header Length.1"]).mean()
    print(f"Fwd Header Length vs .1 duplicate agreement: {100*dup_check:.2f}%")
    full_df = full_df.drop(columns=["Fwd Header Length.1"])
    print(f"Columns after dropping 'Fwd Header Length.1': {len(full_df.columns)}")
else:
    print(" 'Fwd Header Length.1' not found in columns, no direct drop needed (or name is subtly different).")

# Define feature_cols_raw after all known duplicate columns have been handled
feature_cols_raw = [c for c in full_df.columns if c not in NON_FEATURE_COLUMNS + ["Label"]]

Columns before checking for drop: 80
full_df.columns (repr):
  'Destination Port'
  'Flow Duration'
  'Total Fwd Packets'
  'Total Backward Packets'
  'Total Length of Fwd Packets'
  'Total Length of Bwd Packets'
  'Fwd Packet Length Max'
  'Fwd Packet Length Min'
  'Fwd Packet Length Mean'
  'Fwd Packet Length Std'
  'Bwd Packet Length Max'
  'Bwd Packet Length Min'
  'Bwd Packet Length Mean'
  'Bwd Packet Length Std'
  'Flow Bytes/s'
  'Flow Packets/s'
  'Flow IAT Mean'
  'Flow IAT Std'
  'Flow IAT Max'
  'Flow IAT Min'
  'Fwd IAT Total'
  'Fwd IAT Mean'
  'Fwd IAT Std'
  'Fwd IAT Max'
  'Fwd IAT Min'
  'Bwd IAT Total'
  'Bwd IAT Mean'
  'Bwd IAT Std'
  'Bwd IAT Max'
  'Bwd IAT Min'
  'Fwd PSH Flags'
  'Bwd PSH Flags'
  'Fwd URG Flags'
  'Bwd URG Flags'
  'Fwd Header Length'
  'Bwd Header Length'
  'Fwd Packets/s'
  'Bwd Packets/s'
  'Min Packet Length'
  'Max Packet Length'
  'Packet Length Mean'
  'Packet Length Std'
  'Packet Length Variance'
  'FIN Flag Count'
  'SYN Flag Count'


In [ ]:
before = len(full_df)

In [ ]:
full_df[feature_cols_raw] = full_df[feature_cols_raw].replace([np.inf, -np.inf], np.nan)

In [ ]:
n_inf_rows = full_df[feature_cols_raw].isna().any(axis=1).sum()

In [ ]:
print(f"Rows with Infinity/NaN in features: {n_inf_rows} ({100*n_inf_rows/before:.3f}%)")

Rows with Infinity/NaN in features: 0 (0.000%)


In [ ]:
full_df = full_df.dropna(subset=feature_cols_raw)

In [ ]:
after_nan_drop = len(full_df)

In [ ]:
print(f"After dropping Inf/NaN rows: {before:,} -> {after_nan_drop:,}")

After dropping Inf/NaN rows: 2,520,798 -> 2,520,798


In [ ]:
before_dedup = len(full_df)

In [ ]:
full_df = full_df.drop_duplicates()

In [ ]:
after_dedup = len(full_df)

In [ ]:
print(f"After dropping duplicate rows: {before_dedup:,} -> {after_dedup:,} "
      f"(removed {before_dedup - after_dedup:,})")

After dropping duplicate rows: 2,520,798 -> 2,520,798 (removed 0)


## Normalize label strings and Inspect the raw class distribution

Label values across the 8 files are not always byte-identical (dash vs en-dash, double spaces).

Normalize before grouping so "Web Attack - Brute Force" and
"Web Attack \x96 Brute Force" do not silently become two different classes.

In [ ]:
def normalize_label(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s) # collapse repeated whitespace
    s = s.replace("\x96", "-").replace("–", "-")   # normalize dash variants

    return s

In [ ]:
full_df["Label"] = full_df["Label"].apply(normalize_label)

In [ ]:
raw_counts = full_df["Label"].value_counts()

In [ ]:
print("Raw class distribution (after normalization):")
for label, count in raw_counts.items():
    print(f"  {label:35s} {count:>10,}")

Raw class distribution (after normalization):
  BENIGN                               2,095,057
  DoS Hulk                               172,846
  DDoS                                   128,014
  PortScan                                90,694
  DoS GoldenEye                           10,286
  FTP-Patator                              5,931
  DoS slowloris                            5,385
  DoS Slowhttptest                         5,228
  SSH-Patator                              3,219
  Bot                                      1,948
  Web Attack � Brute Force                 1,470
  Web Attack � XSS                           652
  Infiltration                                36
  Web Attack � Sql Injection                  21
  Heartbleed                                  11


## Group into attack categories, then merge rare classes into "Other"

Two-level labeling: a coarse category (what the LLM/tool contract exposes) and the original fine-grained label (kept in metadata for reference, not used as the model target — too many classes have too few samples to learn or stratify reliably), it needs special attention during ETL pipeline.

In [ ]:
CATEGORY_MAP = {
    "benign": "BENIGN",
    "ddos": "DDoS"   # DDoS != DoS
    "dos": "DoS",
    "portscan": "PortScan",
    "bot": "Bot",
    "infiltration": "Infiltration",
    "heartbleed": "Heartbleed",
    "patator": "BruteForce",
    "web attack": "WebAttack",
}

In [ ]:
def to_category(raw_label: str) -> str:
    lowered = raw_label.lower()
    # Sort by key length descending so more specific substrings
    # are checked before shorter ones that could be false-positive substrings
    # of them (e.g. "dos" is literally contained inside "ddos").
    for key, category in sorted(CATEGORY_MAP.items(), key=lambda kv: -len(kv[0])):
        if key in lowered:
            return category
    return "Other"

In [ ]:
full_df["category"] = full_df["Label"].apply(to_category)

In [ ]:
cat_counts = full_df["category"].value_counts()

In [ ]:
print("Category distribution (before rare-class merge):")
for cat, count in cat_counts.items():
    print(f"  {cat:15s} {count:>10,}  ({100*count/len(full_df):5.2f}%)")

Category distribution (before rare-class merge):
  BENIGN           2,095,057  (83.11%)
  DoS                193,745  ( 7.69%)
  DDoS               128,014  ( 5.08%)
  PortScan            90,694  ( 3.60%)
  BruteForce           9,150  ( 0.36%)
  WebAttack            2,143  ( 0.09%)
  Bot                  1,948  ( 0.08%)
  Infiltration            36  ( 0.00%)
  Heartbleed              11  ( 0.00%)


In [ ]:
unrecognized = full_df[full_df["category"] == "Other"]["Label"].unique()

In [ ]:
if len(unrecognized) > 0:
    print(f"\nLabels falling into 'Other' via CATEGORY_MAP (not matched to a named category):")
    for u in unrecognized:
        print(f"  {u!r}")

2,095,057 + 193,745 + 128,014 + 90,694 + 9,150 + 2,143 + 1,948 + 36 + 11 = 2,520,798

##  Merge rare categories below MIN_CLASS_COUNT

Heartbleed-scale classes (~11 samples) cannot be split into train/val/test with any stratification guarantee. Merging them into "Other" is the honest choice, a model  cannot learn a reliable decision boundary from a dozen examples regardless of architecture, and pretending otherwise would produce a metric that looks fine on paper and means nothing.

In [ ]:
final_counts = full_df["category"].value_counts()

In [ ]:
rare_categories = final_counts[final_counts < MIN_CLASS_COUNT].index.tolist()

In [ ]:
print(f"Categories below MIN_CLASS_COUNT={MIN_CLASS_COUNT}, merged into 'Other':")
for c in rare_categories:
    print(f"  {c:15s} {final_counts[c]:>6,} samples")

Categories below MIN_CLASS_COUNT=200, merged into 'Other':
  Other               47 samples


In [ ]:
full_df.loc[full_df["category"].isin(rare_categories), "category"] = "Other"

In [ ]:
final = full_df["category"].value_counts()

In [ ]:
print("Final category distribution:")
for cat, count in final.items():
    print(f"  {cat:15s} {count:>10,}  ({100*count/len(full_df):5.2f}%)")


Final category distribution:
  BENIGN           2,095,057  (83.11%)
  DoS                193,745  ( 7.69%)
  DDoS               128,014  ( 5.08%)
  PortScan            90,694  ( 3.60%)
  BruteForce           9,150  ( 0.36%)
  WebAttack            2,143  ( 0.09%)
  Bot                  1,948  ( 0.08%)
  Other                   47  ( 0.00%)


In [ ]:
full_df["is_attack"] = (full_df["category"] != "BENIGN").astype(int)

In [ ]:
# Binary target, independent of how many attack categories survive.
print(f"\nBinary distribution: BENIGN {int((1 - full_df['is_attack']).sum()):,} | "
      f"ATTACK {int(full_df['is_attack'].sum()):,}")


Binary distribution: BENIGN 2,095,057 | ATTACK 425,741


## Inspect a real feature row before scaling

In [ ]:
# Confirms dtypes are numeric and nothing survived the cleaning as an object/string.
# The feature_cols should explicitly exclude the label columns
feature_cols = [c for c in full_df.columns if c not in NON_FEATURE_COLUMNS + ["Label", "category", "is_attack", "Fwd Header Length.1"]]

In [ ]:
print(f"Feature count: {len(feature_cols)}")
print(f"\nSample row:\n{full_df[feature_cols].iloc[0]}")

Feature count: 77

Sample row:
Destination Port               54865.0
Flow Duration                      3.0
Total Fwd Packets                  2.0
Total Backward Packets             0.0
Total Length of Fwd Packets       12.0
                                ...   
Active Min                         0.0
Idle Mean                          0.0
Idle Std                           0.0
Idle Max                           0.0
Idle Min                           0.0
Name: 0, Length: 77, dtype: float64


In [ ]:
non_numeric = full_df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()

In [ ]:
assert not non_numeric, f

## Encode Labels

LabelEncoder sorts alphabetically by default, which puts "Other" in the middle (O comes before P, W).

 I want "Other" last since it's the catch-all bucket for
merged rare classes, not a real attack category — build the mapping manually.

In [ ]:
present_categories = full_df["category"].unique().tolist()

In [ ]:
ordered_classes = sorted([c for c in present_categories if c != "Other"])

In [ ]:
if "Other" in present_categories:
    ordered_classes.append("Other")

In [ ]:
class_to_idx = {name: i for i, name in enumerate(ordered_classes)}

In [ ]:
class_names = ordered_classes

In [ ]:
y_multiclass = full_df["category"].map(class_to_idx).to_numpy(dtype=np.int64)
y_binary = full_df["is_attack"].to_numpy(dtype=np.int32)

In [ ]:
print(f"Classes (index -> name): {dict(enumerate(class_names))}")

Classes (index -> name): {0: 'BENIGN', 1: 'Bot', 2: 'BruteForce', 3: 'DDoS', 4: 'DoS', 5: 'PortScan', 6: 'WebAttack', 7: 'Other'}


In [ ]:
X = full_df[feature_cols].to_numpy(dtype=np.float32)

In [ ]:
print(f"X shape: {X.shape}")

X shape: (2520798, 77)


Classes (index -> name): {0: 'BENIGN', 1: 'Bot', 2: 'BruteForce', 3: 'DDoS', 4: 'DoS', 5: 'PortScan', 6: 'WebAttack', 7: 'Other'}

##  Stratified split BEFORE Scaling
Avoiding Any Data Leakage...

In [ ]:
X_temp, X_test, ym_temp, ym_test, yb_temp, yb_test = train_test_split(
    X, y_multiclass, y_binary,
    test_size=TEST_SIZE,
    stratify=y_multiclass,
    random_state=SEED
)

In [ ]:
X_train, X_val, ym_train, ym_val, yb_train, yb_val = train_test_split(
    X_temp, ym_temp,
    yb_temp,
    test_size=VAL_SIZE,
    stratify=ym_temp,
    random_state=SEED
)

In [ ]:
for name, y in [("train", ym_train), ("val", ym_val), ("test", ym_test)]:
    n = len(y)
    counts = np.bincount(y, minlength=len(class_names))
    print(f"{name}: {n:,} rows")
    for i, c in enumerate(counts):
        if c > 0:
            print(f"    {class_names[i]:15s} {c:>8,}  ({100*c/n:5.2f}%)")

train: 1,928,410 rows
    BENIGN          1,602,718  (83.11%)
    Bot                1,490  ( 0.08%)
    BruteForce         7,000  ( 0.36%)
    DDoS              97,931  ( 5.08%)
    DoS              148,214  ( 7.69%)
    PortScan          69,381  ( 3.60%)
    WebAttack          1,640  ( 0.09%)
    Other                 36  ( 0.00%)
val: 340,308 rows
    BENIGN           282,833  (83.11%)
    Bot                  263  ( 0.08%)
    BruteForce         1,235  ( 0.36%)
    DDoS              17,282  ( 5.08%)
    DoS               26,156  ( 7.69%)
    PortScan          12,244  ( 3.60%)
    WebAttack            289  ( 0.08%)
    Other                  6  ( 0.00%)
test: 252,080 rows
    BENIGN           209,506  (83.11%)
    Bot                  195  ( 0.08%)
    BruteForce           915  ( 0.36%)
    DDoS              12,801  ( 5.08%)
    DoS               19,375  ( 7.69%)
    PortScan           9,069  ( 3.60%)
    WebAttack            214  ( 0.08%)
    Other                  5  ( 0.00%)


In [ ]:
# Duration/rate features (Flow Duration, Active/Idle *, IAT *, *Bytes/s, *Packets/s) are
# heavy-tailed with a near-zero IQR for the majority of short flows.
LOG_TRANSFORM_COLS = [c for c in feature_cols if any(
    kw in c for kw in ("Duration", "IAT", "Active", "Idle", "Bytes/s", "Packets/s")
)]
print(f"Applying log1p to {len(LOG_TRANSFORM_COLS)} heavy-tailed columns: {LOG_TRANSFORM_COLS}")
RobustScaler divides
# by that near-zero IQR and explodes a handful of long-duration flows to values in the
# hundreds of millions, which is what happened above. log1p compresses that tail before
# scaling, which is standard practice for network flow duration/rate features.
for col_set, name in [(X_train, "train"), (X_val, "val"), (X_test, "test")]:
    pass  # placeholder, real application below on the arrays directly

Applying log1p to 27 heavy-tailed columns: ['Flow Duration', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Packets/s', 'Bwd Packets/s', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']


In [ ]:
#  Scale features (RobustScaler: network flow features are heavy-tailed,
#  with legitimate extreme values like flow duration or byte counts
log_idx = [feature_cols.index(c) for c in LOG_TRANSFORM_COLS] # use np.log1p give us much better scales
X_train[:, log_idx] = np.log1p(np.clip(X_train[:, log_idx], a_min=0, a_max=None))
X_val[:, log_idx] = np.log1p(np.clip(X_val[:, log_idx], a_min=0, a_max=None))
X_test[:, log_idx] = np.log1p(np.clip(X_test[:, log_idx], a_min=0, a_max=None))

In [ ]:
scaler = RobustScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

In [ ]:
print(f"Post-scale train stats: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
print(f"Post-scale value range: [{X_train_scaled.min():.2f}, {X_train_scaled.max():.2f}]")

Post-scale train stats: mean=-4.8254, std=27757.0371
Post-scale value range: [-309733024.00, 661405.56]


## Winsorize (clip to train percentiles), then scale

In [ ]:
LOWER_PCTILE, UPPER_PCTILE = 0.1, 99.9

train_lo = np.percentile(X_train, LOWER_PCTILE, axis=0)
train_hi = np.percentile(X_train, UPPER_PCTILE, axis=0)

def winsorize(X):
    return np.clip(X, train_lo, train_hi)

X_train_clipped = winsorize(X_train)
X_val_clipped = winsorize(X_val)
X_test_clipped = winsorize(X_test)

n_clipped = np.sum((X_train < train_lo) | (X_train > train_hi))
print(f"Train values clipped: {n_clipped:,} of {X_train.size:,} ({100*n_clipped/X_train.size:.4f}%)")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_clipped).astype(np.float32)
X_val_scaled = scaler.transform(X_val_clipped).astype(np.float32)
X_test_scaled = scaler.transform(X_test_clipped).astype(np.float32)

print(f"Post-scale train stats: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
print(f"Post-scale value range: [{X_train_scaled.min():.2f}, {X_train_scaled.max():.2f}]")

Train values clipped: 97,100 of 148,487,570 (0.0654%)
Post-scale train stats: mean=0.9782, std=9.9140
Post-scale value range: [-2.86, 787.67]


##  Save tensors + metadata to LOCAL disk first

In [ ]:
np.save(os.path.join(OUT_DIR, "X_train.npy"), X_train_scaled)
np.save(os.path.join(OUT_DIR, "y_train_multiclass.npy"), ym_train)
np.save(os.path.join(OUT_DIR, "y_train_binary.npy"), yb_train)

In [ ]:
np.save(os.path.join(OUT_DIR, "X_val.npy"), X_val_scaled)
np.save(os.path.join(OUT_DIR, "y_val_multiclass.npy"), ym_val)
np.save(os.path.join(OUT_DIR, "y_val_binary.npy"), yb_val)

In [ ]:
np.save(os.path.join(OUT_DIR, "X_test.npy"), X_test_scaled)
np.save(os.path.join(OUT_DIR, "y_test_multiclass.npy"), ym_test)
np.save(os.path.join(OUT_DIR, "y_test_binary.npy"), yb_test)

In [ ]:
import pickle
# X_train_scaled, X_val_scaled, X_test_scaled

In [ ]:

with open(os.path.join(OUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)


In [ ]:

metadata = {
    "dataset": "CICIDS2017 (chethuhn/network-intrusion-dataset mirror)",
    "role": "NetGuard — auxiliary network-flow classifier feeding structured scores "
            "to the LLM orchestrator",
    "task": "multi-class attack category + binary attack/benign, tabular CNN1D (no embedding layer)",
    "feature_columns": feature_cols,
    "n_features": len(feature_cols),
    "class_names": class_names,
    "min_class_count_threshold": MIN_CLASS_COUNT,
    "categories_merged_into_other": rare_categories,
    "unrecognized_raw_labels": unrecognized.tolist() if len(unrecognized) else [],
    "scaler": "RobustScaler (fit on train only)",
    "seed": SEED,
    "counts": {
        "raw_rows_all_files": int(before),
        "after_inf_nan_drop": int(after_nan_drop),
        "after_dedup": int(after_dedup),
        "train": int(len(ym_train)),
        "val": int(len(ym_val)),
        "test": int(len(ym_test)),
    },
    "class_distribution_train": {
        class_names[i]: int(c) for i, c in enumerate(np.bincount(ym_train, minlength=len(class_names)))
    },
}

In [ ]:
with open(os.path.join(OUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
print("Saved to local disk:")
for fn in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6
    print(f"  {fn}  ({size:.2f} MB)")

Saved to local disk:
  .ipynb_checkpoints  (0.00 MB)
  X_test.npy  (77.64 MB)
  X_train.npy  (593.95 MB)
  X_val.npy  (104.81 MB)
  asdfghj.pkl  (0.00 MB)
  metadata.json  (0.00 MB)
  scaler.pkl  (0.00 MB)
  y_test_binary.npy  (1.01 MB)
  y_test_multiclass.npy  (2.02 MB)
  y_train_binary.npy  (7.71 MB)
  y_train_multiclass.npy  (15.43 MB)
  y_val_binary.npy  (1.36 MB)
  y_val_multiclass.npy  (2.72 MB)


##  Copy to Drive in ONE block

In [ ]:
import shutil

In [ ]:
os.makedirs(DRIVE_OUT, exist_ok=True)

# List all files previously saved to OUT_DIR
files_to_copy = [
    "X_train.npy", "y_train_multiclass.npy", "y_train_binary.npy",
    "X_val.npy", "y_val_multiclass.npy", "y_val_binary.npy",
    "X_test.npy", "y_test_multiclass.npy", "y_test_binary.npy",
    "scaler.pkl", "metadata.json"
]

In [ ]:
# Copy each file from OUT_DIR to DRIVE_OUT of Colab VM
print(f"Copying processed files from {OUT_DIR} to {DRIVE_OUT}...")
for filename in files_to_copy:
    src_path = os.path.join(OUT_DIR, filename)
    dst_path = os.path.join(DRIVE_OUT, filename)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"  Copied {filename}")
    else:
        print(f"  Warning: {filename} not found in {OUT_DIR}, skipping copy.")

print("Copy complete.")

# Now load the files from DRIVE_OUT
Xtr = np.load(os.path.join(DRIVE_OUT, "X_train.npy"))
ymtr = np.load(os.path.join(DRIVE_OUT, "y_train_multiclass.npy"))
ybtr = np.load(os.path.join(DRIVE_OUT, "y_train_binary.npy"))

Copying processed files from /content/netguard_data/processed to /content/drive/MyDrive/Aegis-CyberSec-Guard/netguard/dataV1...
  Copied X_train.npy
  Copied y_train_multiclass.npy
  Copied y_train_binary.npy
  Copied X_val.npy
  Copied y_val_multiclass.npy
  Copied y_val_binary.npy
  Copied X_test.npy
  Copied y_test_multiclass.npy
  Copied y_test_binary.npy
  Copied scaler.pkl
  Copied metadata.json
Copy complete.


In [ ]:
with open(os.path.join(DRIVE_OUT, "metadata.json")) as f:
    meta = json.load(f)

In [ ]:
assert Xtr.shape[1] == meta["n_features"], "feature count mismatch"
assert Xtr.shape[0] == ymtr.shape[0] == ybtr.shape[0], "X/y length mismatch"
assert not np.isnan(Xtr).any(), "NaN survived into the saved tensor"
assert not np.isinf(Xtr).any(), "Infinity survived into the saved tensor"

In [ ]:
print("Reload verification passed.")
print(f"  X_train: {Xtr.shape}")
print(f"  classes: {meta['class_names']}")
print(f"  binary attack rate in train: {100*ybtr.mean():.2f}%")
print("\nETL complete. Tensors ready for the NetGuard CNN1D training script.")

Reload verification passed.
  X_train: (1928410, 77)
  classes: ['BENIGN', 'Bot', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'WebAttack', 'Other']
  binary attack rate in train: 16.89%

ETL complete. Tensors ready for the NetGuard CNN1D training script.


In [ ]:
!ls -lah /content/drive/MyDrive/Aegis-CyberSec-Guard/netguard/dataV1

total 770M
-rw------- 1 root root 2.8K Aug 19 16:45 metadata.json
-rw------- 1 root root 1.4K Aug 19 16:45 scaler.pkl
-rw------- 1 root root  75M Aug 19 16:45 X_test.npy
-rw------- 1 root root 567M Aug 19 16:45 X_train.npy
-rw------- 1 root root 100M Aug 19 16:45 X_val.npy
-rw------- 1 root root 985K Aug 19 16:45 y_test_binary.npy
-rw------- 1 root root 2.0M Aug 19 16:45 y_test_multiclass.npy
-rw------- 1 root root 7.4M Aug 19 16:45 y_train_binary.npy
-rw------- 1 root root  15M Aug 19 16:45 y_train_multiclass.npy
-rw------- 1 root root 1.3M Aug 19 16:45 y_val_binary.npy
-rw------- 1 root root 2.6M Aug 19 16:45 y_val_multiclass.npy


In [ ]:
import numpy as np

In [ ]:
col_std = X_train_scaled.std(axis=0)
col_max = np.abs(X_train_scaled).max(axis=0)
worst_idx = np.argsort(-col_max)[:10]

print("Columnas con los valores escalados más extremos:")
for i in worst_idx:
    print(f"  {feature_cols[i]:35s} max_abs={col_max[i]:.2e}  std={col_std[i]:.2e}")

Columnas con los valores escalados más extremos:
  Total Length of Bwd Packets         max_abs=7.88e+02  std=3.51e+01
  Subflow Bwd Bytes                   max_abs=7.88e+02  std=3.51e+01
  Init_Win_bytes_backward             max_abs=2.78e+02  std=3.77e+01
  Destination Port                    max_abs=1.67e+02  std=4.89e+01
  Packet Length Variance              max_abs=1.61e+02  std=1.67e+01
  Total Length of Fwd Packets         max_abs=1.26e+02  std=6.89e+00
  Subflow Fwd Bytes                   max_abs=1.26e+02  std=6.89e+00
  Subflow Bwd Packets                 max_abs=1.12e+02  std=5.40e+00
  Total Backward Packets              max_abs=1.12e+02  std=5.40e+00
  Bwd Header Length                   max_abs=1.04e+02  std=4.99e+00
